# Law Firm Operations — Pre-Filled Configuration
### Ready-to-Run Config for the Law Firms Use Case

This is a **pre-configured** version of `00_Industry_Config` specifically for the
**Law Firm Documentation Burden** demo. All table lists, artifact names,
and data paths are hardcoded from the existing 23-CSV law firm dataset.

**Use this instead of `00_Industry_Config` if you want to skip auto-discovery and run
directly with the known law firm tables.**

---

### Data Summary
- **6 Dimensions** — Attorneys, Clients, Matters, Legal Task Types, Practice Groups, Courts
- **6 Batch Facts** — Time Entries, Discovery Events, Attorney Wellness, Work Product Quality, Client Satisfaction, Matter Performance
- **6 Event Facts** → Eventhouse — DMS Interactions, Filing Events, Contract Events, Matter Transfers, Deadline Alerts, Billing Events
- **5 Streams** → Eventhouse — DMS Activity, Time Tracking, Court Deadlines, Discovery Progress, Client Communications

### Ontology
- **6 Entity Types:** Attorney, Client, Matter, LegalTaskType, PracticeGroup, Court
- **5 Relationship Types:** works_on_matter, represents_client, member_of_group, files_in_court, assigned_task
- **6 Contextualizations:** LegalDocEvent, DMSInteraction, FilingEvent, ContractEvent, DeadlineAlert, BillingEvent

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# INDUSTRY SETTING
# ════════════════════════════════════════════════════════════════════════

INDUSTRY       = "law_firms"
INDUSTRY_LABEL = "Law Firms"

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# FABRIC ARTIFACT NAMES
# ════════════════════════════════════════════════════════════════════════
# Update these to match your Fabric workspace artifact names.

LAKEHOUSE_NAME      = "LawFirmLakehouse"
WAREHOUSE_NAME      = "LawFirm_Datawarehouse"
EVENTHOUSE_NAME     = "lawfirm_rt_store"
ONTOLOGY_NAME       = "LawFirmOpsOntology"
DATA_AGENT_NAME     = "LawFirmQA"
OPS_AGENT_NAME      = "LawFirmDocBurden"
SEMANTIC_MODEL_NAME = "Law_Firm_Ops_Model"

print(f"Lakehouse:      {LAKEHOUSE_NAME}")
print(f"Warehouse:      {WAREHOUSE_NAME}")
print(f"Eventhouse:     {EVENTHOUSE_NAME}")
print(f"Ontology:       {ONTOLOGY_NAME}")
print(f"Data Agent:     {DATA_AGENT_NAME}")
print(f"Ops Agent:      {OPS_AGENT_NAME}")
print(f"Semantic Model: {SEMANTIC_MODEL_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# DATA PATHS & EVENTHOUSE CONNECTION
# ════════════════════════════════════════════════════════════════════════

# CSV files location in Lakehouse Files area
CSV_BASE_PATH = "/lakehouse/default/Files/law_firm_operations/data"

# Schemas
LAKEHOUSE_SCHEMA = "dbo"
WAREHOUSE_SCHEMA = "dbo"

# ── Eventhouse Connection ───────────────────────────────────────────────
# Fill these in from your Fabric workspace
EVENTHOUSE_CLUSTER_URI = ""   # e.g. "https://trd-xxxxx.z5.kusto.fabric.microsoft.com"
EVENTHOUSE_DATABASE    = ""   # e.g. "lawfirm_rt_store"

# Ontology package path
ONTOLOGY_PACKAGE_PATH = "/lakehouse/default/Files/law_firm_ops_ontology_package.iq"

print(f"CSV source:     {CSV_BASE_PATH}")
print(f"LH schema:      {LAKEHOUSE_SCHEMA}")
print(f"WH schema:      {WAREHOUSE_SCHEMA}")
print(f"Eventhouse:     {EVENTHOUSE_CLUSTER_URI or '(fill in your cluster URI)'}")
print(f"Ontology pkg:   {ONTOLOGY_PACKAGE_PATH}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LAW FIRM TABLE DEFINITIONS (pre-filled, no auto-discovery needed)
# ════════════════════════════════════════════════════════════════════════

import os, json

# ── 6 Dimension Tables → Lakehouse + Warehouse ──────────────────────────
DIM_TABLES = [
    "dim_attorneys",
    "dim_clients",
    "dim_matters",
    "dim_legal_task_types",
    "dim_practice_groups",
    "dim_courts",
]

# ── 6 Batch Fact Tables → Lakehouse + Warehouse ─────────────────────────
FACT_TABLES_BATCH = [
    "fact_time_entries",
    "fact_discovery_events",
    "fact_attorney_wellness",
    "fact_work_product_quality",
    "fact_client_satisfaction",
    "fact_matter_performance",
]

# ── 6 Event Fact Tables → Eventhouse (KQL) ──────────────────────────────
FACT_TABLES_EVENT = [
    "fact_dms_interactions",
    "fact_filing_events",
    "fact_contract_events",
    "fact_matter_transfers",
    "fact_deadline_alerts",
    "fact_billing_events",
]

# ── 5 Streaming Tables → Eventhouse (KQL) ───────────────────────────────
STREAM_TABLES = [
    "stream_dms_activity",
    "stream_time_tracking",
    "stream_court_deadlines",
    "stream_discovery_progress",
    "stream_client_communications",
]

# ── Combined Target Lists ───────────────────────────────────────────────
LAKEHOUSE_TABLES  = DIM_TABLES + FACT_TABLES_BATCH
WAREHOUSE_TABLES  = DIM_TABLES + FACT_TABLES_BATCH + FACT_TABLES_EVENT
EVENTHOUSE_TABLES = FACT_TABLES_EVENT + STREAM_TABLES

# ── KQL Table Name Mapping (CSV name → KQL table name) ─────────────────
# Event facts strip the 'fact_' prefix; streams strip 'stream_' prefix
EVENTHOUSE_KQL_NAMES = {
    "fact_dms_interactions":          "dms_interactions",
    "fact_filing_events":             "filing_events",
    "fact_contract_events":           "contract_events",
    "fact_matter_transfers":          "matter_transfers",
    "fact_deadline_alerts":           "deadline_alerts",
    "fact_billing_events":            "billing_events",
    "stream_dms_activity":            "dms_activity",
    "stream_time_tracking":           "time_tracking",
    "stream_court_deadlines":         "court_deadlines",
    "stream_discovery_progress":      "discovery_progress",
    "stream_client_communications":   "client_communications",
}

print(f"{'='*60}")
print(f"LAW FIRM TABLE INVENTORY")
print(f"{'='*60}")
print(f"\nDimension tables ({len(DIM_TABLES)}):")
for t in DIM_TABLES: print(f"  • {t}")
print(f"\nFact tables — Batch ({len(FACT_TABLES_BATCH)}):")
for t in FACT_TABLES_BATCH: print(f"  • {t}")
print(f"\nFact tables — Event/Eventhouse ({len(FACT_TABLES_EVENT)}):")
for t in FACT_TABLES_EVENT: print(f"  • {t}  →  {EVENTHOUSE_KQL_NAMES[t]}")
print(f"\nStreaming tables — Eventhouse ({len(STREAM_TABLES)}):")
for t in STREAM_TABLES: print(f"  • {t}  →  {EVENTHOUSE_KQL_NAMES[t]}")
print(f"\n{'─'*60}")
print(f"Lakehouse target:  {len(LAKEHOUSE_TABLES)} tables (12 Delta tables)")
print(f"Warehouse target:  {len(WAREHOUSE_TABLES)} tables (18 SQL tables)")
print(f"Eventhouse target: {len(EVENTHOUSE_TABLES)} tables (11 KQL tables)")
print(f"Total:             23 CSV files")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# EXPECTED ROW COUNTS (for validation in downstream notebooks)
# ════════════════════════════════════════════════════════════════════════

EXPECTED_ROW_COUNTS = {
    # Dimensions
    "dim_attorneys":                 30,
    "dim_clients":                   25,
    "dim_matters":                   40,
    "dim_legal_task_types":           20,
    "dim_practice_groups":             8,
    "dim_courts":                     12,
    # Batch Facts
    "fact_time_entries":             500,
    "fact_discovery_events":         150,
    "fact_attorney_wellness":         30,
    "fact_work_product_quality":     200,
    "fact_client_satisfaction":       25,
    "fact_matter_performance":        80,
    # Event Facts
    "fact_dms_interactions":         500,
    "fact_filing_events":             60,
    "fact_contract_events":           80,
    "fact_matter_transfers":          20,
    "fact_deadline_alerts":          100,
    "fact_billing_events":            60,
    # Streams
    "stream_dms_activity":            80,
    "stream_time_tracking":           80,
    "stream_court_deadlines":         80,
    "stream_discovery_progress":      80,
    "stream_client_communications":   80,
}

total_lakehouse = sum(EXPECTED_ROW_COUNTS[t] for t in LAKEHOUSE_TABLES)
total_eventhouse = sum(EXPECTED_ROW_COUNTS[t] for t in EVENTHOUSE_TABLES)
print(f"Expected Lakehouse rows:  {total_lakehouse:,}")
print(f"Expected Eventhouse rows: {total_eventhouse:,}")
print(f"Expected total rows:      {sum(EXPECTED_ROW_COUNTS.values()):,}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SCHEMA INSPECTION HELPER
# ════════════════════════════════════════════════════════════════════════

def preview_table(table_name, base_path=CSV_BASE_PATH, rows=5):
    """Read a CSV and display schema + sample rows."""
    path = f"{base_path}/{table_name}.csv"
    df = spark.read.option("header", True).option("inferSchema", True).csv(path)
    print(f"\n{'─'*60}")
    print(f"Table: {table_name}  |  {df.count()} rows  |  {len(df.columns)} columns")
    print(f"{'─'*60}")
    df.printSchema()
    df.show(rows, truncate=False)
    return df

# Uncomment to preview specific tables:
# preview_table("dim_attorneys")
# preview_table("fact_time_entries")
# preview_table("stream_court_deadlines")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CONFIG SUMMARY (exported to downstream notebooks via %run)
# ════════════════════════════════════════════════════════════════════════

CONFIG = {
    "industry":             INDUSTRY,
    "industry_label":       INDUSTRY_LABEL,
    "lakehouse_name":       LAKEHOUSE_NAME,
    "warehouse_name":       WAREHOUSE_NAME,
    "eventhouse_name":      EVENTHOUSE_NAME,
    "eventhouse_uri":       EVENTHOUSE_CLUSTER_URI,
    "eventhouse_db":        EVENTHOUSE_DATABASE,
    "ontology_name":        ONTOLOGY_NAME,
    "ontology_package":     ONTOLOGY_PACKAGE_PATH,
    "data_agent_name":      DATA_AGENT_NAME,
    "ops_agent_name":       OPS_AGENT_NAME,
    "semantic_model_name":  SEMANTIC_MODEL_NAME,
    "csv_base_path":        CSV_BASE_PATH,
    "lakehouse_schema":     LAKEHOUSE_SCHEMA,
    "warehouse_schema":     WAREHOUSE_SCHEMA,
    "dim_tables":           DIM_TABLES,
    "fact_tables_batch":    FACT_TABLES_BATCH,
    "fact_tables_event":    FACT_TABLES_EVENT,
    "stream_tables":        STREAM_TABLES,
    "lakehouse_tables":     LAKEHOUSE_TABLES,
    "warehouse_tables":     WAREHOUSE_TABLES,
    "eventhouse_tables":    EVENTHOUSE_TABLES,
}

print(f"\n{'═'*60}")
print(f"✅ LAW FIRM CONFIG READY")
print(f"{'═'*60}")
print(json.dumps({k: v if not isinstance(v, list) else f"{len(v)} tables" for k, v in CONFIG.items()}, indent=2))
print(f"\nThis config is imported by downstream notebooks via:")
print(f"  %run ./LawFirms_Config")